# B1 · Transfer learning et modèles pré-entraînés — notebook d'exercices

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookIA/blob/main/chapitres/b1_transfer/03_notebook.ipynb)

La partie 0 vérifie tes réponses courtes aux quiz, aux rappels et aux exercices papier. Chaque exercice de code : un énoncé, une cellule à compléter (les `...` et les `raise NotImplementedError`), puis une cellule de vérification (`wb.check`, ou des tests). « Tout exécuter » (*Run all*) va jusqu'au bout même si rien n'est rempli : ce qui n'est pas fait affiche ⏳. Les questions « dans tes notes » qui n'ont pas de cellule 📝 se notent dans la section « Notes sur le notebook » de ta copie de `06_mes_reponses.md`. Bloqué 15 minutes ? `04_indices.md`.

> Travaille dans **ta copie** (`mon_travail/b1_transfer/03_notebook.ipynb`, créée par `python tools/start_chapter.py B1`) : ce fichier-ci est mis à jour par Claude. Sur Colab, le badge ouvre cette version du dépôt, qui n'est pas enregistrée : crée puis ouvre ta copie comme l'explique `00_setup/COLAB.md` §2.

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | B1.Q1 à B1.Q11, B1.R1 à B1.R3, B1.1 à B1.5 | vérifier tes réponses courtes | 🧠 🔁 ✏️ | ★ à ★★ | |
| A | B1.11 à B1.16 | charger et sonder : trois modèles pré-entraînés, un ResNet face aux images de 32 pixels, embeddings figés, linear probe et k plus proches voisins, résolution, projections, sondage bloc par bloc (*prochaine session de génération*) | | | |
| B | B1.17 à B1.19 | fine-tuner : le dernier bloc avec des learning rates par groupe, un fine-tuning qui détruit le modèle, le fine-tuning complet (*prochaine session de génération*) | | | |
| C | B1.20 à B1.23 | quand le transfert aide ou échoue : courbes d'apprentissage, domaine lointain, oubli catastrophique (*prochaine session de génération*) | | | |
| D | B1.24 à B1.28 | modèles de fondation, coût et livraison : zéro-shot avec CLIP, few-shot par prototypes, latence sur CPU, publication du modèle dérivé, et le défi (*prochaine session de génération*) | | | |

In [ ]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (CPU, a few minutes); False = full version
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookIA.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookIA")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WB_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wb").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wb").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier du workbook.")

os.environ["WB_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wb  # noqa: E402

cfg = wb.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="B1")  # your own library

## Objectifs et rappel express

- Charger un modèle pré-entraîné avec son prétraitement (torchvision, Hugging Face Hub) et lire sa model card et sa licence.
- Entraîner un linear probe et un k plus proches voisins sur des embeddings figés, et sonder les blocs d'un backbone.
- Fine-tuner un réseau : dégel progressif, learning rates par groupe, warmup, batchnorm maîtrisée.
- Mesurer le gain du pré-entraînement par des courbes d'apprentissage, et diagnostiquer transfert négatif et oubli catastrophique.
- Classer en zéro-shot et en few-shot avec CLIP et DINOv2, et comparer des backbones sur CPU.

**Rappel express.** Linear probe : $\hat{\mathbf{y}} = \operatorname{softmax}(\mathbf{W}\,\phi(\mathbf{x}) + \mathbf{b})$, $d \times K + K$ paramètres. Learning rates discriminatifs : $\eta_\ell = \eta_{\text{tête}}\,\gamma^{L-\ell}$ ; warmup : $\eta_t = \eta_{\max}\,\min(1, t / t_{\text{warmup}})$. L2-SP : $\Omega = \frac{\lambda}{2}\lVert\theta - \theta_0\rVert^2$. Zéro-shot : $p(c \mid \mathbf{x}) \propto \exp(s \cos(\mathbf{u}_x, \mathbf{v}_c))$. Prototype : $\mathbf{m}_c = \frac{1}{|S_c|}\sum_{\mathbf{x} \in S_c} \phi(\mathbf{x})$.

## Partie 0 · Vérifier tes réponses courtes (quiz, rappels, ✏️ B1.1 à B1.5)

Fais d'abord les quiz, les rappels et les exercices de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse courte ici : **la valeur** que tu as trouvée, pas l'expression Python, sinon tu ne vérifies rien. Un nombre : `42` ou `0.3` (en Python, le séparateur décimal est un **point** ; `0,3` sans guillemets serait un couple de deux nombres) ; un vrai ou faux : `True` ou `False` ; un choix : la lettre seule, entre guillemets (`"B"`) ; plusieurs lettres : entre guillemets (`"AC"`) ; une forme de tenseur : entre guillemets (`"(2, 3)"`) ; un nom de classe : entre guillemets (`"chat"`). Arrondis comme l'énoncé le demande, et seulement à la fin du calcul. Les réponses pas encore remplies affichent ⏳. Les questions « dans ta copie », la dérivation ∂ B1.6, l'estimation 🧮 B1.7, la réflexion (🗣️ B1.8, ⚖️ B1.9, 📄 B1.10) et l'entretien se corrigent avec `05_solutions.md`.

**B1.Q1 — Réutiliser plutôt que repartir de zéro : trois raisons et une limite**

In [ ]:
# B1.Q1: the values you found on paper
answer_B1_Q1a = ...  # a) the letters, e.g. "AC"
answer_B1_Q1b = ...  # b) a whole number
answer_B1_Q1c = ...  # c) True or False
answer_B1_Q1d = ...  # d) the letter of your choice, e.g. "B"
answer_B1_Q1e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_B1_Q1a, answer_B1_Q1b, answer_B1_Q1c, answer_B1_Q1d, answer_B1_Q1e]):
    wb.check(f"B1.Q1{letter}", answer)

**B1.Q2 — Premières couches génériques, dernières couches spécialisées : vrai ou faux**

In [ ]:
# B1.Q2: the values you found on paper
answer_B1_Q2a = ...  # a) True or False
answer_B1_Q2b = ...  # b) True or False
answer_B1_Q2c = ...  # c) True or False
answer_B1_Q2d = ...  # d) True or False
answer_B1_Q2e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_B1_Q2a, answer_B1_Q2b, answer_B1_Q2c, answer_B1_Q2d, answer_B1_Q2e]):
    wb.check(f"B1.Q2{letter}", answer)

**B1.Q3 — Où trouver un modèle pré-entraîné, et quoi lire avant de s'en servir**

In [ ]:
# B1.Q3: the values you found on paper
answer_B1_Q3a = ...  # a) the letter of your choice, e.g. "B"
answer_B1_Q3b = ...  # b) the letters, e.g. "AC"
answer_B1_Q3c = ...  # c) True or False
answer_B1_Q3d = ...  # d) the letter of your choice, e.g. "B"
answer_B1_Q3e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_B1_Q3a, answer_B1_Q3b, answer_B1_Q3c, answer_B1_Q3d, answer_B1_Q3e]):
    wb.check(f"B1.Q3{letter}", answer)

**B1.Q4 — weights.transforms() : pourquoi le prétraitement fait partie du modèle**

In [ ]:
# B1.Q4: the values you found on paper
answer_B1_Q4a = ...  # a) the letter of your choice, e.g. "B"
answer_B1_Q4b = ...  # b) True or False
answer_B1_Q4c = ...  # c) a whole number
answer_B1_Q4d = ...  # d) the letter of your choice, e.g. "B"
answer_B1_Q4e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_B1_Q4a, answer_B1_Q4b, answer_B1_Q4c, answer_B1_Q4d, answer_B1_Q4e]):
    wb.check(f"B1.Q4{letter}", answer)

**B1.Q5 — Extraction de features, linear probe, fine-tuning : qui s'entraîne ?**

In [ ]:
# B1.Q5: the values you found on paper
answer_B1_Q5a = ...  # a) the letter of your choice, e.g. "B"
answer_B1_Q5b = ...  # b) the letter of your choice, e.g. "B"
answer_B1_Q5c = ...  # c) True or False
answer_B1_Q5d = ...  # d) a whole number
answer_B1_Q5e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_B1_Q5a, answer_B1_Q5b, answer_B1_Q5c, answer_B1_Q5d, answer_B1_Q5e]):
    wb.check(f"B1.Q5{letter}", answer)

**B1.Q6 — Pourquoi un learning rate 10 à 100 fois plus petit pour fine-tuner ?**

In [ ]:
# B1.Q6: the values you found on paper
answer_B1_Q6a = ...  # a) the letter of your choice, e.g. "B"
answer_B1_Q6b = ...  # b) True or False
answer_B1_Q6c = ...  # c) the letter of your choice, e.g. "B"
answer_B1_Q6d = ...  # d) True or False
answer_B1_Q6e = ...  # e) the letter of your choice, e.g. "B"

for letter, answer in zip("abcde", [answer_B1_Q6a, answer_B1_Q6b, answer_B1_Q6c, answer_B1_Q6d, answer_B1_Q6e]):
    wb.check(f"B1.Q6{letter}", answer)

**B1.Q7 — Batchnorm pendant le fine-tuning sur de tout petits batches**

In [ ]:
# B1.Q7: the values you found on paper
answer_B1_Q7a = ...  # a) the letter of your choice, e.g. "B"
answer_B1_Q7b = ...  # b) True or False
answer_B1_Q7c = ...  # c) True or False
answer_B1_Q7d = ...  # d) the letter of your choice, e.g. "B"
answer_B1_Q7e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_B1_Q7a, answer_B1_Q7b, answer_B1_Q7c, answer_B1_Q7d, answer_B1_Q7e]):
    wb.check(f"B1.Q7{letter}", answer)

**B1.Q8 — Peu ou beaucoup de données, domaine proche ou lointain : la bonne stratégie**

In [ ]:
# B1.Q8: the values you found on paper
answer_B1_Q8a = ...  # a) the letter of your choice, e.g. "B"
answer_B1_Q8b = ...  # b) the letter of your choice, e.g. "B"
answer_B1_Q8c = ...  # c) the letter of your choice, e.g. "B"
answer_B1_Q8d = ...  # d) True or False
answer_B1_Q8e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_B1_Q8a, answer_B1_Q8b, answer_B1_Q8c, answer_B1_Q8d, answer_B1_Q8e]):
    wb.check(f"B1.Q8{letter}", answer)

**B1.Q9 — Transfert négatif et oubli catastrophique : reconnaître les symptômes**

In [ ]:
# B1.Q9: the values you found on paper
answer_B1_Q9a = ...  # a) the letter of your choice, e.g. "B"
answer_B1_Q9b = ...  # b) the letter of your choice, e.g. "B"
answer_B1_Q9c = ...  # c) True or False
answer_B1_Q9d = ...  # d) True or False
answer_B1_Q9e = ...  # e) the letter of your choice, e.g. "B"

for letter, answer in zip("abcde", [answer_B1_Q9a, answer_B1_Q9b, answer_B1_Q9c, answer_B1_Q9d, answer_B1_Q9e]):
    wb.check(f"B1.Q9{letter}", answer)

**B1.Q10 — Classer sans exemple d'entraînement : le zéro-shot de CLIP**

In [ ]:
# B1.Q10: the values you found on paper
answer_B1_Q10a = ...  # a) the letter of your choice, e.g. "B"
answer_B1_Q10b = ...  # b) True or False
answer_B1_Q10c = ...  # c) True or False
answer_B1_Q10d = ...  # d) the letter of your choice, e.g. "B"
answer_B1_Q10e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_B1_Q10a, answer_B1_Q10b, answer_B1_Q10c, answer_B1_Q10d, answer_B1_Q10e]):
    wb.check(f"B1.Q10{letter}", answer)

**B1.Q11 — Auto-supervisé (DINOv2) ou supervisé sur ImageNet : quels embeddings ?**

In [ ]:
# B1.Q11: the values you found on paper
answer_B1_Q11a = ...  # a) the letter of your choice, e.g. "B"
answer_B1_Q11b = ...  # b) True or False
answer_B1_Q11c = ...  # c) the letter of your choice, e.g. "B"
answer_B1_Q11d = ...  # d) True or False
answer_B1_Q11e = ...  # e) a whole number

for letter, answer in zip("abcde", [answer_B1_Q11a, answer_B1_Q11b, answer_B1_Q11c, answer_B1_Q11d, answer_B1_Q11e]):
    wb.check(f"B1.Q11{letter}", answer)

**B1.R1 — Ch. 29 : licence d'un dataset et décalage de distribution**

In [ ]:
# B1.R1: the values you found on paper
answer_B1_R1a = ...  # a) True or False
answer_B1_R1b = ...  # b) the letter of your choice, e.g. "B"
answer_B1_R1c = ...  # c) the letter of your choice, e.g. "B"
answer_B1_R1d = ...  # d) True or False
answer_B1_R1e = ...  # e) the letter of your choice, e.g. "B"

for letter, answer in zip("abcde", [answer_B1_R1a, answer_B1_R1b, answer_B1_R1c, answer_B1_R1d, answer_B1_R1e]):
    wb.check(f"B1.R1{letter}", answer)

**B1.R2 — Ch. 23 : geler un réseau et remplacer sa tête en PyTorch**

In [ ]:
# B1.R2: the values you found on paper
answer_B1_R2a = ...  # a) the letter of your choice, e.g. "B"
answer_B1_R2b = ...  # b) True or False
answer_B1_R2c = ...  # c) a whole number
answer_B1_R2d = ...  # d) True or False
answer_B1_R2e = ...  # e) the letter of your choice, e.g. "B"

for letter, answer in zip("abcde", [answer_B1_R2a, answer_B1_R2b, answer_B1_R2c, answer_B1_R2d, answer_B1_R2e]):
    wb.check(f"B1.R2{letter}", answer)

**B1.R3 — Ch. 21 : ce que détectent les filtres des premières et des dernières couches**

In [ ]:
# B1.R3: the values you found on paper
answer_B1_R3a = ...  # a) the letter of your choice, e.g. "B"
answer_B1_R3b = ...  # b) a whole number
answer_B1_R3c = ...  # c) True or False
answer_B1_R3d = ...  # d) True or False
answer_B1_R3e = ...  # e) the letter of your choice, e.g. "B"

for letter, answer in zip("abcde", [answer_B1_R3a, answer_B1_R3b, answer_B1_R3c, answer_B1_R3d, answer_B1_R3e]):
    wb.check(f"B1.R3{letter}", answer)

**Ex B1.1 — Nouvelle tête : formes des tenseurs de l'image 32×32 redimensionnée jusqu'aux logits**

In [ ]:
# Ex B1.1: the values you found on paper
answer_B1_1a = ...  # a) a shape as a string, e.g. "(2, 3)"
answer_B1_1b = ...  # b) a shape as a string, e.g. "(2, 3)"
answer_B1_1c = ...  # c) a shape as a string, e.g. "(2, 3)"
answer_B1_1d = ...  # d) a shape as a string, e.g. "(2, 3)"
answer_B1_1e = ...  # e) a shape as a string, e.g. "(2, 3)"
answer_B1_1f = ...  # f) a whole number
answer_B1_1g = ...  # g) a whole number
answer_B1_1h = ...  # h) True or False

for letter, answer in zip("abcdefgh", [answer_B1_1a, answer_B1_1b, answer_B1_1c, answer_B1_1d, answer_B1_1e, answer_B1_1f, answer_B1_1g, answer_B1_1h]):
    wb.check(f"B1.1{letter}", answer)

**Ex B1.2 — k plus proches voisins cosinus sur six embeddings donnés**

In [ ]:
# Ex B1.2: the values you found on paper
answer_B1_2a = ...  # a) a number, 3 decimals
answer_B1_2b = ...  # b) a vector number, from 0 to 5
answer_B1_2c = ...  # c) a class name as a string, e.g. "chat"
answer_B1_2d = ...  # d) a class name as a string, e.g. "chat"
answer_B1_2e = ...  # e) a vector number, from 0 to 5
answer_B1_2f = ...  # f) a number, 3 decimals
answer_B1_2g = ...  # g) True or False

for letter, answer in zip("abcdefg", [answer_B1_2a, answer_B1_2b, answer_B1_2c, answer_B1_2d, answer_B1_2e, answer_B1_2f, answer_B1_2g]):
    wb.check(f"B1.2{letter}", answer)

**Ex B1.3 — Learning rates discriminatifs et warmup : le learning rate de chaque groupe, pas à pas**

In [ ]:
# Ex B1.3: the values you found on paper
answer_B1_3a = ...  # a) a number, 1 decimal
answer_B1_3b = ...  # b) a whole number
answer_B1_3c = ...  # c) a whole number
answer_B1_3d = ...  # d) a number, 2 decimals
answer_B1_3e = ...  # e) a number, 2 decimals
answer_B1_3f = ...  # f) a number, 2 decimals
answer_B1_3g = ...  # g) True or False
answer_B1_3h = ...  # h) True or False

for letter, answer in zip("abcdefgh", [answer_B1_3a, answer_B1_3b, answer_B1_3c, answer_B1_3d, answer_B1_3e, answer_B1_3f, answer_B1_3g, answer_B1_3h]):
    wb.check(f"B1.3{letter}", answer)

**Ex B1.4 — Compter les paramètres entraînables : tête seule, dernier bloc, réseau entier**

In [ ]:
# Ex B1.4: the values you found on paper
answer_B1_4a = ...  # a) a whole number
answer_B1_4b = ...  # b) a whole number
answer_B1_4c = ...  # c) a whole number
answer_B1_4d = ...  # d) a whole number
answer_B1_4e = ...  # e) a whole number
answer_B1_4f = ...  # f) a number, 1 decimal
answer_B1_4g = ...  # g) a whole number
answer_B1_4h = ...  # h) True or False

for letter, answer in zip("abcdefgh", [answer_B1_4a, answer_B1_4b, answer_B1_4c, answer_B1_4d, answer_B1_4e, answer_B1_4f, answer_B1_4g, answer_B1_4h]):
    wb.check(f"B1.4{letter}", answer)

**Ex B1.5 — Zéro-shot CLIP à la main : cosinus, logit_scale et softmax**

In [ ]:
# Ex B1.5: the values you found on paper
answer_B1_5a = ...  # a) a number, 3 decimals
answer_B1_5b = ...  # b) a number, 3 decimals
answer_B1_5c = ...  # c) a number, 3 decimals
answer_B1_5d = ...  # d) a number, 4 decimals
answer_B1_5e = ...  # e) a number, 2 decimals
answer_B1_5f = ...  # f) True or False

for letter, answer in zip("abcdef", [answer_B1_5a, answer_B1_5b, answer_B1_5c, answer_B1_5d, answer_B1_5e, answer_B1_5f]):
    wb.check(f"B1.5{letter}", answer)

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Sais-tu suivre les formes d'un mini-batch dans un ResNet pré-entraîné et compter ce que tu entraînes selon la stratégie ?
2. Sais-tu écrire les groupes de paramètres et le planning d'un fine-tuning, et dire pourquoi les premières couches bougent le moins ?
3. Sais-tu calculer les probabilités d'un zéro-shot de CLIP, et dire ce que fait l'échelle $s$ ?
4. Sais-tu dire ce qu'un modèle dérivé hérite de son modèle de base (licence, biais, limites) ?

**Pour aller plus loin** : le tutoriel de PyTorch sur le transfer learning et l'article de Yosinski et coll. (2014), cités dans la fiche. La suite : le bonus B2, tokenisation et embeddings.

*Les exercices de code B1.11 à B1.28 (parties A à D : charger et sonder, fine-tuner, quand le transfert aide ou échoue, modèles de fondation, coût et livraison) seront ajoutés à la prochaine session de génération ; relance alors `python tools/start_chapter.py B1` pour obtenir le notebook complet.*